# 07 — Scala vs PySpark: Choosing Your Spark Language

> **Note:** This notebook runs on the **Almond Scala kernel** (`name: almond`).  
> All code cells are Scala. Make sure you have selected the `Scala (Spark 3.5)` kernel before running.

## When to Choose Scala vs Python

Spark is written in Scala, which means Scala is the "native tongue" of the framework.
However, PySpark has become the dominant language for Spark users — primarily because of
the Python data science ecosystem.

This notebook explores the trade-offs and shows that the DataFrame/Dataset API looks
nearly identical in both languages.

### API Symmetry

For most day-to-day Spark work the APIs are a direct translation of each other:

```python
# PySpark
df.filter(col("age") > 30).groupBy("dept").count()
```
```scala
// Scala
df.filter(col("age") > 30).groupBy("dept").count()
```

The syntax is almost identical. The real differences emerge at the edges: type safety,
custom extensions, and integration with domain-specific libraries.

### Compilation Safety

Scala is **statically typed** and compiled. This means:
- Column name typos are caught **at compile time** when using the typed `Dataset` API.
- Python only catches such errors **at runtime** — sometimes deep inside a long job.

**Cluster:** `spark://spark-master:7077`  
**Data dir:** `/home/jovyan/data`

In [ ]:
// -----------------------------------------------------------------------
// Almond-spark: the bridge between the Almond Scala kernel and Spark.
//
// $ivy magic imports pull JARs from Maven Central into the notebook
// classpath at kernel startup — no sbt/Maven project needed.
// -----------------------------------------------------------------------

import $ivy.`sh.almond::almond-spark:0.14.0`

import org.apache.spark.sql._
import org.apache.spark.sql.functions._
import almond.spark._

// NotebookSparkSession is Almond's wrapper around SparkSession.builder.
// It routes Spark logs through the Almond display system so they appear
// cleanly in the notebook output rather than flooding stderr.
val spark: SparkSession = (
  NotebookSparkSession.builder()
    .master("spark://spark-master:7077")
    .appName("07-Scala-Comparison")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

// Import implicits so we can use $"columnName" syntax and toDS() / toDF()
import spark.implicits._

spark.sparkContext.setLogLevel("WARN")

println(s"Spark version : ${spark.version}")
println(s"App name      : ${spark.sparkContext.appName}")
println(s"Master        : ${spark.sparkContext.master}")

## PySpark vs Scala — Side-by-Side for the 10 Most Common Operations

| Operation | PySpark | Scala |
|---|---|---|
| **Create DataFrame from list** | `spark.createDataFrame([(1,"a")], ["id","name"])` | `Seq((1,"a")).toDF("id","name")` |
| **Select columns** | `df.select("id", "name")` | `df.select("id", "name")` |
| **Filter rows** | `df.filter(col("age") > 30)` | `df.filter($"age" > 30)` |
| **Add a column** | `df.withColumn("x", col("a")+1)` | `df.withColumn("x", $"a"+1)` |
| **GroupBy + agg** | `df.groupBy("dept").agg(avg("sal"))` | `df.groupBy($"dept").agg(avg($"sal"))` |
| **Join** | `df1.join(df2, "id", "inner")` | `df1.join(df2, Seq("id"), "inner")` |
| **Sort** | `df.orderBy(col("sal").desc())` | `df.orderBy($"sal".desc)` |
| **UDF definition** | `@udf(returnType=StringType())` | `val f = udf((x:String) => x.upper)` |
| **Read Parquet** | `spark.read.parquet("/path")` | `spark.read.parquet("/path")` |
| **Typed Dataset** | Not available (DataFrame only) | `df.as[MyCaseClass]` |

> The `$"column"` syntax in Scala is syntactic sugar for `col("column")` — it is enabled
> by `import spark.implicits._`. Both forms are valid in Scala; `col()` is available without the import.

In [ ]:
// -----------------------------------------------------------------------
// Typed Dataset with a case class
//
// In Scala you can go beyond DataFrame (which is Dataset[Row]) and create
// a Dataset[Employee] — a strongly-typed collection where each element
// is an Employee object, not an anonymous Row.
//
// Benefits:
//   • .filter and .map use real Scala lambdas — no Column expression needed
//   • Column name typos become compile-time errors (within the typed API)
//   • IDE autocompletion works on field names
// -----------------------------------------------------------------------

// Define the data model as a case class.
// Spark derives an Encoder automatically for case classes — no extra work needed.
case class Employee(
  id:         Int,
  name:       String,
  department: String,
  salary:     Double,
  age:        Int
)

// Create a Seq of Employee objects and convert to a typed Dataset[Employee]
val employees: Dataset[Employee] = Seq(
  Employee(1,  "Alice",   "Engineering", 95000.0,  32),
  Employee(2,  "Bob",     "Engineering", 87000.0,  28),
  Employee(3,  "Carol",   "Marketing",   72000.0,  41),
  Employee(4,  "Dave",    "Marketing",   68000.0,  35),
  Employee(5,  "Eve",     "Engineering", 110000.0, 45),
  Employee(6,  "Frank",   "HR",          65000.0,  29),
  Employee(7,  "Grace",   "HR",          70000.0,  38),
  Employee(8,  "Heidi",   "Engineering", 92000.0,  31),
  Employee(9,  "Ivan",    "Marketing",   75000.0,  27),
  Employee(10, "Judy",    "HR",          69000.0,  44)
).toDS()   // toDS() uses the implicit Encoder[Employee] derived from the case class

println(s"Dataset type: ${employees.getClass.getSimpleName}")
println(s"Schema:")
employees.printSchema()

// ---- Typed transformations ------------------------------------------------
// These lambdas work on actual Employee objects — full type safety.

// filter: keep only engineers with salary > 90k
val seniorEngineers: Dataset[Employee] = employees
  .filter(emp => emp.department == "Engineering" && emp.salary > 90000.0)

println("\nSenior Engineers (salary > 90k):")
seniorEngineers.show()

// map: extract a new case class with just name + adjusted salary
case class SalaryBand(name: String, band: String)

val withBand: Dataset[SalaryBand] = employees.map { emp =>
  val band = if (emp.salary >= 90000) "Senior" else if (emp.salary >= 70000) "Mid" else "Junior"
  SalaryBand(emp.name, band)
}

println("Salary bands:")
withBand.show()

In [ ]:
// -----------------------------------------------------------------------
// Equivalent aggregation to the beginner notebook (03)
// groupBy + agg — same logic, Scala syntax.
//
// Compare with the PySpark version:
//   df.groupBy("department").agg(
//       count("*").alias("headcount"),
//       round(avg("salary"), 2).alias("avg_salary"),
//       max("salary").alias("max_salary")
//   ).orderBy(desc("avg_salary"))
// The Scala version below is almost character-for-character identical.
// -----------------------------------------------------------------------

// Switch from typed Dataset[Employee] back to untyped DataFrame for
// column-expression-based aggregations (the common production pattern)
val empDF: DataFrame = employees.toDF()

val deptStats: DataFrame = empDF
  .groupBy($"department")
  .agg(
    count("*").alias("headcount"),
    round(avg($"salary"), 2).alias("avg_salary"),
    max($"salary").alias("max_salary"),
    min($"age").alias("youngest_age")
  )
  .orderBy($"avg_salary".desc)

println("Department statistics:")
deptStats.show(truncate = false)

// Adding a derived column — identical to PySpark withColumn
val withBonus: DataFrame = empDF
  .withColumn("bonus", round($"salary" * 0.10, 2))
  .withColumn("total_comp", round($"salary" + $"bonus", 2))

println("Employee compensation with bonus:")
withBonus.select("name", "department", "salary", "bonus", "total_comp").show()

In [ ]:
// -----------------------------------------------------------------------
// Compile-time safety demonstration
//
// The typed Dataset API catches column name errors at COMPILE TIME.
// The DataFrame API catches them at RUNTIME — just like PySpark.
//
// This is the most important practical difference between the two APIs.
// -----------------------------------------------------------------------

println("=== Typed Dataset API — compile-time safety ===")
println()

// CORRECT — accessing a real field; this compiles
val avgSalaryTyped: Double = employees
  .map(_.salary)       // _.salary is a real Scala field reference
  .reduce(_ + _) / employees.count()

println(f"Average salary (typed): $$avgSalaryTyped%.2f")

// If you tried:  employees.map(_.salry)   ← typo
// The Scala compiler would immediately report:
//   error: value salry is not a member of Employee
// You would never even be able to submit the job.

println()
println("=== DataFrame API — runtime error on bad column name ===")
println()

// The DataFrame API uses string column names — just like PySpark.
// A typo here will only fail when Spark tries to resolve the query plan.
try {
  // Intentional typo: 'salery' instead of 'salary'
  val badDF = empDF.select($"salery")   // AnalysisException at runtime
  badDF.show()
} catch {
  case e: org.apache.spark.sql.AnalysisException =>
    println(s"[RUNTIME ERROR caught] ${e.getMessage.take(120)}...")
    println()
    println("In PySpark this would be exactly the same — AnalysisException at runtime.")
    println("In the typed Dataset API, the equivalent would fail at compile time.")
}

println()
println("Takeaway: Use Dataset[T] when compile-time safety matters more than API flexibility.")
println("Use DataFrame when you need column-expression DSL, SQL, or ML pipeline compatibility.")

## When Scala Wins

### 1. JVM-Native Execution — No Serialization Overhead
PySpark DataFrame operations are executed on JVM executors. However, Python UDFs (non-Pandas)
require row-by-row data serialization between JVM and the Python process. Scala UDFs run
entirely inside the JVM — **zero serialization overhead**.

### 2. Custom Encoders and Efficient Serialization
Scala lets you write custom `Encoder[T]` implementations, which control exactly how objects
are serialized in the Tungsten memory format. This is impossible from Python.

### 3. Writing Spark Extensions and Custom Data Sources
If you need to write a custom Catalyst optimizer rule, a custom `DataSourceV2` connector,
or a custom `Aggregator`, you must use Scala (or Java). These APIs are not exposed to Python.

### 4. Streaming with Complex State (mapGroupsWithState)
The stateful streaming API `mapGroupsWithState` and `flatMapGroupsWithState` are available
in Scala/Java with full type safety. The Python equivalent is less ergonomic and has
higher latency due to serialization.

### 5. Compile-Time Correctness at Scale
Large teams benefit enormously from compile-time checks catching errors in CI/CD pipelines
before a job even reaches the cluster. Scala's type system prevents entire categories of
bugs that Python users only discover after a multi-hour job fails.

## When PySpark Wins

### 1. Data Science and ML Libraries
The Python ecosystem is unmatched for ML/AI: `scikit-learn`, `PyTorch`, `TensorFlow`,
`XGBoost`, `LightGBM`, `statsmodels`, and hundreds more. Calling these from PySpark
(via `pandas_udf` or `applyInPandas`) is natural. Doing the same from Scala requires
subprocess calls or dedicated JVM libraries.

### 2. Faster Iteration and Prototyping
Python's dynamic nature, REPL, and the Jupyter ecosystem make exploratory data analysis
faster. You can prototype in a PySpark notebook and only rewrite performance-critical
sections in Scala if needed.

### 3. Pandas API on Spark (`pyspark.pandas`)
Spark 3.2+ ships a Pandas-compatible API (`pyspark.pandas`, formerly Koalas) that lets
existing Pandas code run on a Spark cluster with minimal changes. There is no Scala
equivalent of this convenience layer.

### 4. Hiring Pool and Team Expertise
The global pool of Python engineers vastly exceeds Scala engineers. For most organizations,
Python is the lower-risk choice for team velocity and onboarding.

### 5. Notebooks and Visualization
Libraries like `matplotlib`, `seaborn`, `plotly`, and `bokeh` integrate natively with
Jupyter. While Almond (this kernel) enables Scala notebooks, the PySpark + Jupyter
ecosystem has far more tooling.

---

### Decision Framework

| Scenario | Recommendation |
|---|---|
| Data exploration, ad-hoc analysis | PySpark |
| ML feature engineering + model training | PySpark |
| Production ETL pipeline with strict SLAs | Scala |
| Writing a Spark library or data source connector | Scala |
| Team already uses Python across the stack | PySpark |
| Large Spark-native engineering team | Scala |
| Streaming with complex stateful logic | Scala |
| Quick prototype to validate an idea | PySpark |

In [ ]:
// -----------------------------------------------------------------------
// Clean up — stop the SparkSession to release cluster resources.
// -----------------------------------------------------------------------

println(s"Active jobs before stop: ${spark.sparkContext.statusTracker.getActiveJobIds().length}")

spark.stop()

println("SparkSession stopped. Resources released.")